In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('installments_payments.csv')

In [ ]:
df.shape

(13605401, 8)

In [ ]:
df.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB


In [ ]:
df.duplicated().sum()

np.int64(0)

Check **"SK_ID_CURR"** and **"SK_ID_PREV"**

In [ ]:
print("Rows:", len(df))

print("Unique customers :",
      df['SK_ID_CURR'].nunique())

print("Unique previous accounts :",
      df['SK_ID_PREV'].nunique())

print("SK_ID_PREV unique :",
      df['SK_ID_PREV'].is_unique)

Rows: 13605401
Unique customers : 339587
Unique previous accounts : 997752
SK_ID_PREV unique : False


MISSING PERCENTAGE

In [ ]:
missing_pct = (
    df.isna()
    .mean()
    .mul(100)
    .round(4)
    .sort_values(ascending=False)
)

missing_pct

,0
AMT_PAYMENT,0.0214
DAYS_ENTRY_PAYMENT,0.0214
SK_ID_PREV,0.0000
SK_ID_CURR,0.0000
NUM_INSTALMENT_NUMBER,0.0000
NUM_INSTALMENT_VERSION,0.0000
DAYS_INSTALMENT,0.0000
AMT_INSTALMENT,0.0000


In [ ]:
df.isnull().sum()

,0
SK_ID_PREV,0
SK_ID_CURR,0
NUM_INSTALMENT_VERSION,0
NUM_INSTALMENT_NUMBER,0
DAYS_INSTALMENT,0
DAYS_ENTRY_PAYMENT,2905
AMT_INSTALMENT,0
AMT_PAYMENT,2905


In [ ]:
# ============================================================
# 1. PAYMENT BEHAVIOR FEATURES
# ============================================================

# Payment delay:
# Positive = late, 0 = on time, negative = early
df['PAYMENT_DELAY'] = (
    df['DAYS_ENTRY_PAYMENT'] - df['DAYS_INSTALMENT']
)

# Amount that was scheduled but not paid
df['PAYMENT_SHORTFALL'] = (
    df['AMT_INSTALMENT'] - df['AMT_PAYMENT']
).clip(lower=0)

# Late payment flag
df['IS_LATE_PAYMENT'] = (
    df['PAYMENT_DELAY'] > 0
).astype(int)

# Underpaid installment flag
df['IS_UNDERPAID'] = (
    df['PAYMENT_SHORTFALL'] > 0
).astype(int)

This section defines key payment behavior features:

*   **PAYMENT_DELAY**: Calculates if a payment was early, on time, or late.

*   **PAYMENT_SHORTFALL**: Quantifies any unpaid amount from a scheduled installment.

*   **IS_LATE_PAYMENT**: A flag indicating whether a payment was made late.

*   **IS_UNDERPAID**: A flag indicating whether the full installment amount was paid.

In [ ]:
# ============================================================
# 2. LOAN-LEVEL AGGREGATION
# ============================================================

loan_level = (
    df
    .groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(

        # Number of installments observed for this loan
        LOAN_INSTALLMENTS=(
            'NUM_INSTALMENT_NUMBER', 'count'
        ),

        # Scheduled payment amount
        LOAN_TOTAL_DUE=(
            'AMT_INSTALMENT', 'sum'
        ),

        # Actual amount paid
        LOAN_TOTAL_PAID=(
            'AMT_PAYMENT', 'sum'
        ),

        # Typical payment size
        LOAN_AVG_PAYMENT=(
            'AMT_PAYMENT', 'mean'
        ),

        # Payment timing
        LOAN_AVG_DELAY=(
            'PAYMENT_DELAY', 'mean'
        ),

        # Worst late-payment episode
        LOAN_MAX_LATE_DAYS=(
            'PAYMENT_DELAY', 'max'
        ),

        # Number of late installments
        LOAN_LATE_COUNT=(
            'IS_LATE_PAYMENT', 'sum'
        ),

        # Total amount underpaid
        LOAN_TOTAL_SHORTFALL=(
            'PAYMENT_SHORTFALL', 'sum'
        ),

        # Number of underpaid installments
        LOAN_UNDERPAID_COUNT=(
            'IS_UNDERPAID', 'sum'
        )
    )
    .reset_index()
)

In [ ]:
# ============================================================
# 3. LOAN-LEVEL RATIOS
# ============================================================

# How much of the scheduled amount was actually paid?
loan_level['LOAN_PAYMENT_RATIO'] = (
    loan_level['LOAN_TOTAL_PAID']
    /
    loan_level['LOAN_TOTAL_DUE'].replace(0, np.nan)
)

# What proportion of installments were late?
loan_level['LOAN_LATE_RATE'] = (
    loan_level['LOAN_LATE_COUNT']
    /
    loan_level['LOAN_INSTALLMENTS'].replace(0, np.nan)
)

# What proportion of installments were underpaid?
loan_level['LOAN_UNDERPAID_RATE'] = (
    loan_level['LOAN_UNDERPAID_COUNT']
    /
    loan_level['LOAN_INSTALLMENTS'].replace(0, np.nan)
)

In [ ]:
loan_level.head()

,SK_ID_CURR,SK_ID_PREV,LOAN_INSTALLMENTS,LOAN_TOTAL_DUE,LOAN_TOTAL_PAID,LOAN_AVG_PAYMENT,LOAN_AVG_DELAY,LOAN_MAX_LATE_DAYS,LOAN_LATE_COUNT,LOAN_TOTAL_SHORTFALL,LOAN_UNDERPAID_COUNT,LOAN_PAYMENT_RATIO,LOAN_LATE_RATE,LOAN_UNDERPAID_RATE
0,100001,1369693,4,29250.900,29250.900,7312.725000,-15.500000,-6.0,0,0.0,0,1.0,0.000000,0.0
1,100001,1851984,3,11945.025,11945.025,3981.675000,3.666667,11.0,1,0.0,0,1.0,0.333333,0.0
2,100002,1038818,19,219625.695,219625.695,11559.247105,-20.421053,-12.0,0,0.0,0,1.0,0.000000,0.0
3,100003,1810518,7,1150977.330,1150977.330,164425.332857,-4.428571,-3.0,0,0.0,0,1.0,0.000000,0.0
4,100003,2396755,12,80773.380,80773.380,6731.115000,-6.750000,-1.0,0,0.0,0,1.0,0.000000,0.0


In [ ]:
# ============================================================
# 4. CUSTOMER-LEVEL AGGREGATION
# ============================================================

customer_features = (
    loan_level
    .groupby('SK_ID_CURR')
    .agg(

        # Number of previous loans
        INSTALLMENT_LOANS=(
            'SK_ID_PREV', 'nunique'
        ),

        # Overall repayment amounts
        TOTAL_AMOUNT_DUE=(
            'LOAN_TOTAL_DUE', 'sum'
        ),

        TOTAL_AMOUNT_PAID=(
            'LOAN_TOTAL_PAID', 'sum'
        ),

        # Overall payment behavior
        AVG_PAYMENT_RATIO=(
            'LOAN_PAYMENT_RATIO', 'mean'
        ),

        # Payment timing
        AVG_PAYMENT_DELAY=(
            'LOAN_AVG_DELAY', 'mean'
        ),

        MAX_LATE_PAYMENT_DAYS=(
            'LOAN_MAX_LATE_DAYS', 'max'
        ),

        # Late payment behavior
        TOTAL_LATE_PAYMENTS=(
            'LOAN_LATE_COUNT', 'sum'
        ),

        AVG_LOAN_LATE_RATE=(
            'LOAN_LATE_RATE', 'mean'
        ),

        MAX_LOAN_LATE_RATE=(
            'LOAN_LATE_RATE', 'max'
        ),

        # Underpayment behavior
        TOTAL_PAYMENT_SHORTFALL=(
            'LOAN_TOTAL_SHORTFALL', 'sum'
        ),

        TOTAL_UNDERPAID_INSTALLMENTS=(
            'LOAN_UNDERPAID_COUNT', 'sum'
        ),

        AVG_LOAN_UNDERPAID_RATE=(
            'LOAN_UNDERPAID_RATE', 'mean'
        ),

        MAX_LOAN_UNDERPAID_RATE=(
            'LOAN_UNDERPAID_RATE', 'max'
        )
    )
    .reset_index()
)

In [ ]:
customer_features['PAYMENT_COMPLETION_RATIO'] = (
    customer_features['TOTAL_AMOUNT_PAID']
    /
    customer_features['TOTAL_AMOUNT_DUE'].replace(0, np.nan)
)

In [ ]:
customer_features.head()

,SK_ID_CURR,INSTALLMENT_LOANS,TOTAL_AMOUNT_DUE,TOTAL_AMOUNT_PAID,AVG_PAYMENT_RATIO,AVG_PAYMENT_DELAY,MAX_LATE_PAYMENT_DAYS,TOTAL_LATE_PAYMENTS,AVG_LOAN_LATE_RATE,MAX_LOAN_LATE_RATE,TOTAL_PAYMENT_SHORTFALL,TOTAL_UNDERPAID_INSTALLMENTS,AVG_LOAN_UNDERPAID_RATE,MAX_LOAN_UNDERPAID_RATE,PAYMENT_COMPLETION_RATIO
0,100001,2,41195.925,41195.925,1.0,-5.916667,11.0,1,0.166667,0.333333,0.0,0,0.0,0.0,1.0
1,100002,1,219625.695,219625.695,1.0,-20.421053,-12.0,0,0.000000,0.000000,0.0,0,0.0,0.0,1.0
2,100003,3,1618864.650,1618864.650,1.0,-7.448413,-1.0,0,0.000000,0.000000,0.0,0,0.0,0.0,1.0
3,100004,1,21288.465,21288.465,1.0,-7.666667,-3.0,0,0.000000,0.000000,0.0,0,0.0,0.0,1.0
4,100005,1,56161.845,56161.845,1.0,-23.555556,1.0,1,0.111111,0.111111,0.0,0,0.0,0.0,1.0


In [ ]:
# ============================================================
# 5. VALIDATION
# ============================================================

print("Shape:", customer_features.shape)

print(
    "Unique customers:",
    customer_features['SK_ID_CURR'].nunique()
)

print(
    "Duplicate customers:",
    customer_features['SK_ID_CURR'].duplicated().sum()
)

print("\nMissing values:")
display(
    customer_features
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head()
)

Shape: (339587, 15)
Unique customers: 339587
Duplicate customers: 0

Missing values:


,0
MAX_LATE_PAYMENT_DAYS,9
AVG_PAYMENT_DELAY,9
AVG_PAYMENT_RATIO,3
PAYMENT_COMPLETION_RATIO,3
TOTAL_AMOUNT_DUE,0


**SAVE IT**

In [ ]:
final_installment_features = customer_features

### **CHANGING THE COL_PREFIX (INSTALL)**

| Current Name                   | New Name                               |
| ------------------------------ | -------------------------------------- |
| `SK_ID_CURR`                   | `SK_ID_CURR`                           |
| `INSTALLMENT_LOANS`            | `INSTALL_LOANS`                        |
| `TOTAL_AMOUNT_DUE`             | `INSTALL_TOTAL_AMOUNT_DUE`             |
| `TOTAL_AMOUNT_PAID`            | `INSTALL_TOTAL_AMOUNT_PAID`            |
| `AVG_PAYMENT_RATIO`            | `INSTALL_AVG_PAYMENT_RATIO`            |
| `AVG_PAYMENT_DELAY`            | `INSTALL_AVG_PAYMENT_DELAY`            |
| `MAX_LATE_PAYMENT_DAYS`        | `INSTALL_MAX_LATE_PAYMENT_DAYS`        |
| `TOTAL_LATE_PAYMENTS`          | `INSTALL_TOTAL_LATE_PAYMENTS`          |
| `AVG_LOAN_LATE_RATE`           | `INSTALL_AVG_LOAN_LATE_RATE`           |
| `MAX_LOAN_LATE_RATE`           | `INSTALL_MAX_LOAN_LATE_RATE`           |
| `TOTAL_PAYMENT_SHORTFALL`      | `INSTALL_TOTAL_PAYMENT_SHORTFALL`      |
| `TOTAL_UNDERPAID_INSTALLMENTS` | `INSTALL_TOTAL_UNDERPAID_INSTALLMENTS` |
| `AVG_LOAN_UNDERPAID_RATE`      | `INSTALL_AVG_LOAN_UNDERPAID_RATE`      |
| `MAX_LOAN_UNDERPAID_RATE`      | `INSTALL_MAX_LOAN_UNDERPAID_RATE`      |
| `PAYMENT_COMPLETION_RATIO`     | `INSTALL_PAYMENT_COMPLETION_RATIO`     |


In [ ]:
final_installment_features = final_installment_features.rename(
    columns=lambda col: (
        col.replace("INSTALLMENT_", "INSTALL_", 1)
        if col.startswith("INSTALLMENT_")
        else col if col == "SK_ID_CURR"
        else f"INSTALL_{col}"
    )
)

In [ ]:
final_installment_features.head()

,SK_ID_CURR,INSTALL_LOANS,INSTALL_TOTAL_AMOUNT_DUE,INSTALL_TOTAL_AMOUNT_PAID,INSTALL_AVG_PAYMENT_RATIO,INSTALL_AVG_PAYMENT_DELAY,INSTALL_MAX_LATE_PAYMENT_DAYS,INSTALL_TOTAL_LATE_PAYMENTS,INSTALL_AVG_LOAN_LATE_RATE,INSTALL_MAX_LOAN_LATE_RATE,INSTALL_TOTAL_PAYMENT_SHORTFALL,INSTALL_TOTAL_UNDERPAID_INSTALLMENTS,INSTALL_AVG_LOAN_UNDERPAID_RATE,INSTALL_MAX_LOAN_UNDERPAID_RATE,INSTALL_PAYMENT_COMPLETION_RATIO
0,100001,2,41195.925,41195.925,1.0,-5.916667,11.0,1,0.166667,0.333333,0.0,0,0.0,0.0,1.0
1,100002,1,219625.695,219625.695,1.0,-20.421053,-12.0,0,0.000000,0.000000,0.0,0,0.0,0.0,1.0
2,100003,3,1618864.650,1618864.650,1.0,-7.448413,-1.0,0,0.000000,0.000000,0.0,0,0.0,0.0,1.0
3,100004,1,21288.465,21288.465,1.0,-7.666667,-3.0,0,0.000000,0.000000,0.0,0,0.0,0.0,1.0
4,100005,1,56161.845,56161.845,1.0,-23.555556,1.0,1,0.111111,0.111111,0.0,0,0.0,0.0,1.0


In [27]:
final_installment_features.to_csv(
    'installments_aggregated.csv',
    index=False
)